# AI Lab: Multi-PDF RAG Chatbot using Google Colab
This notebook teaches the complete Retrieval-Augmented Generation (RAG) pipeline using multiple PDFs, FAISS, LangChain and Gemini.

## Learning Objectives
- Understand hallucination and why RAG is needed
- Load multiple PDFs
- Split documents into chunks
- Create embeddings
- Build a FAISS vector database
- Retrieve relevant chunks
- Query Gemini using retrieved context
- Display document sources


# Step 1 - Install Libraries

In [ ]:
!pip -q install langchain langchain-community langchain-google-genai faiss-cpu pypdf sentence-transformers

# Step 2 - Enter Gemini API Key

Create a free API key from Google AI Studio:
https://aistudio.google.com/app/apikey

Paste it below.


In [ ]:
import os
os.environ["GOOGLE_API_KEY"]=input("Enter Gemini API Key: ").strip()


# Step 3 - Upload Multiple PDFs

In [ ]:
from google.colab import files
uploaded=files.upload()

### What happens?
Each uploaded PDF will become part of the chatbot's knowledge base.


# Step 4 - Read all PDFs

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents=[]

for file in os.listdir():
    if file.endswith(".pdf"):
        loader=PyPDFLoader(file)
        docs=loader.load()
        for d in docs:
            d.metadata["source"]=file
        documents.extend(docs)

print("Pages Loaded:",len(documents))


# Step 5 - Chunk Documents

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks=splitter.split_documents(documents)

print("Chunks:",len(chunks))


# Step 6 - Create Embeddings

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings=GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)


# Step 7 - Build FAISS Vector Database

In [ ]:
from langchain_community.vectorstores import FAISS

db=FAISS.from_documents(chunks,embeddings)
print("Vector database created.")


# Step 8 - Create Retriever

In [ ]:
retriever=db.as_retriever(search_kwargs={"k":3})

# Step 9 - Load Gemini LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# Step 10 - Ask Questions

Type **exit** to stop.


In [ ]:
while True:

    question=input("\nAsk a Question: ")

    if question.lower()=="exit":
        break

    docs=retriever.invoke(question)

    print("\nRetrieved Chunks")
    print("="*60)

    context=""

    for d in docs:
        print("Source:",d.metadata["source"])
        print(d.page_content[:250])
        print("-"*60)
        context+=d.page_content+"\n\n"

    prompt=f'''
You are a helpful AI assistant.

Answer ONLY from the supplied context.

If the answer is unavailable in the context, reply:
"I don't have enough information from the uploaded documents."

Context:
{context}

Question:
{question}
'''

    response=llm.invoke(prompt)

    print("\nAnswer")
    print("="*60)
    print(response.content)

    print("\nSources")
    used=set()
    for d in docs:
        if d.metadata["source"] not in used:
            print("-",d.metadata["source"])
            used.add(d.metadata["source"])


# Experiments

1. Change `k` from 3 to 5.
2. Change chunk size to 500 and 1000.
3. Upload different PDFs.
4. Ask questions outside the uploaded documents.
5. Observe retrieved chunks before reading the answer.


# Viva Questions

1. What is hallucination?
2. Why do we split documents?
3. What is chunk overlap?
4. What are embeddings?
5. Why use a vector database?
6. What is semantic search?
7. What does FAISS do?
8. What happens when k increases?
9. Difference between RAG and a normal LLM?
10. Why display document sources?
